# Notebook 03 — FinBERT Embeddings
**Cells 1–9: Run in Google Colab (free T4 GPU)**  
**Cell 10: Run in VS Code after downloading outputs**

---

**What we're doing and why:**  
Our tabular features (amount, hour, distance, etc.) describe *how* a transaction looks. But the merchant name and category describe *where* and *what type* of transaction it is. FinBERT can extract richer semantic meaning from those text fields than a simple label encoding can.

FinBERT is a version of BERT (Google's language model) that was fine-tuned specifically on financial text — news, filings, earnings calls. This makes it understand words like `grocery`, `entertainment`, `shopping_net` in a financial context better than a generic language model would.

The output is a 768-number vector per transaction that captures the semantic meaning of the merchant+category text. We reduce this to 30 numbers with PCA before merging with our tabular features.

## Cell 1 — Install Dependencies (Colab only)
Google Colab has many libraries pre-installed, but we need to install `transformers` and `sentence-transformers` explicitly.

**What is FinBERT?**  
BERT (Bidirectional Encoder Representations from Transformers) is a deep learning model that reads text in both directions simultaneously — left-to-right AND right-to-left — giving it a rich understanding of context. FinBERT is the version trained specifically on financial text by ProsusAI. When we pass it a string like `"merchant: grocery_net, category: grocery_pos"`, it returns a 768-dimensional vector that encodes the semantic meaning of that text in a financial context.

In [2]:
# RUN IN GOOGLE COLAB
# Install required libraries (Colab doesn't have these by default)
!pip install -q transformers torch sentence-transformers scikit-learn

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'GPU available:   {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name:        {torch.cuda.get_device_name(0)}')

PyTorch version: 2.10.0+cu128
GPU available:   True
GPU name:        Tesla T4


## Cell 2 — Upload Data to Colab
We need `fraudTrain.csv` accessible in Colab. The easiest way is to upload directly from your computer.  
**Option A (recommended):** Upload directly — takes 2–3 minutes for 500MB.  
**Option B:** Mount Google Drive if you have the file there already.

In [3]:
import pandas as pd
import numpy as np

In [4]:
# OPTION A: Upload directly from your computer
#from google.colab import files
#print('A file picker will appear. Select fraudTrain.csv from your data/raw/ folder.')
#uploaded = files.upload()  # pick fraudTrain.csv
#df = pd.read_csv('fraudTrain.csv')

# OPTION B: Mount Google Drive (uncomment if file is in Drive)
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/fraud-detection-project/data/raw/fraudTrain.csv')

print(f'Loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

Mounted at /content/drive
Loaded: (1296675, 23)
Columns: ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0


## Cell 2b — HuggingFace Authentication
We load your HuggingFace token from the `.env` file in your project root on Drive.
This avoids rate limits when downloading FinBERT and lets you access gated models.

**Where your token lives:** `fraud-detection-project/.env` on Google Drive  
**Format inside `.env`:** `HF_TOKEN=hf_your_token_here`

> If the token is missing the notebook will still run — public models like FinBERT
> don't require auth, but authenticated requests have higher rate limits.

In [6]:
# Load HuggingFace token from .env file on Google Drive
# The .env file is gitignored — it is never committed to GitHub
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/fraud-detection-project'
env_path   = Path(DRIVE_ROOT) / '.env'
hf_token   = None

if env_path.exists():
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line.startswith('HF_TOKEN=') and not line.startswith('#'):
            hf_token = line.split('=', 1)[1].strip()
            break

if hf_token and hf_token != 'paste_your_huggingface_token_here':
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('HuggingFace authenticated successfully')
else:
    print('WARNING: HF_TOKEN not found or not set in .env')
    print('Proceeding without auth — public models still work but may be rate-limited.')
    hf_token = None


HuggingFace authenticated successfully


## Cell 3 — Build Input Text Strings
FinBERT takes text as input, not raw column values. We combine `merchant` and `category` into a single descriptive string per row.

The format `"merchant: X, category: Y"` is a natural-language style that FinBERT handles well — it was trained on text that looks like this.

In [7]:
# Combine merchant name and category into a single text string per transaction
df['text_input'] = (
    'merchant: ' + df['merchant'].str.replace('fraud_', '', regex=False) +
    ', category: ' + df['category']
)

print(f'Text inputs created for {len(df):,} transactions.')
print(f'\n5 example text inputs:')
for t in df['text_input'].sample(5, random_state=42).values:
    print(f'  "{t}"')

Text inputs created for 1,296,675 transactions.

5 example text inputs:
  "merchant: Towne LLC, category: misc_pos"
  "merchant: Friesen Ltd, category: health_fitness"
  "merchant: Mohr Inc, category: shopping_pos"
  "merchant: Gaylord-Powlowski, category: home"
  "merchant: Christiansen, Goyette and Schamberger, category: gas_transport"


## Cell 4 — Extract Unique Texts Only (Performance Trick)
Running FinBERT on 1.3 million rows would take hours even on a GPU — but most rows share the same merchant+category combination. We only need to run the model once per unique combination, then map the result back to all rows.

This is a classic ML engineering trick: **deduplicate before inference, re-expand after**.

In [8]:
# Get unique text combinations
unique_texts = df['text_input'].unique().tolist()

print(f'Total rows:           {len(df):,}')
print(f'Unique text combos:   {len(unique_texts):,}')
print(f'Compute reduction:    {(1 - len(unique_texts)/len(df))*100:.1f}% fewer FinBERT calls')
print(f'\nSample unique texts:')
for t in unique_texts[:5]:
    print(f'  "{t}"')

Total rows:           1,296,675
Unique text combos:   700
Compute reduction:    99.9% fewer FinBERT calls

Sample unique texts:
  "merchant: Rippin, Kub and Mann, category: misc_net"
  "merchant: Heller, Gutmann and Zieme, category: grocery_pos"
  "merchant: Lind-Buckridge, category: entertainment"
  "merchant: Kutch, Hermiston and Farrell, category: gas_transport"
  "merchant: Keeling-Crist, category: misc_pos"


## Cell 5 — Load FinBERT Model
We load the tokenizer and model from HuggingFace. The tokenizer converts text into token IDs that the model can process. The model outputs hidden states — we extract the `[CLS]` token.

**What is the [CLS] token?**  
BERT prepends every input with a special `[CLS]` (classification) token. During training, this token is designed to accumulate a summary of the entire sentence's meaning. Its hidden state (a 768-number vector) is what we extract and use as the embedding — it represents the full meaning of the input text in one vector.

In [11]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'ProsusAI/finbert'

print(f'Loading {MODEL_NAME}...')
# Pass hf_token for authenticated requests (set in Cell 2b)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
model     = AutoModel.from_pretrained(MODEL_NAME, token=hf_token)

# Move model to GPU if available (much faster than CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
model.eval()  # disable dropout — we are doing inference, not training

print(f'Model loaded on: {device}')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Loading ProsusAI/finbert...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on: cuda
Model parameters: 109,482,240


## Cell 6 — Run FinBERT Inference in Batches
We pass text to the model 64 strings at a time (a "batch"). This is more efficient than one-at-a-time because the GPU can process many inputs in parallel.

**What do the 768 numbers represent?**  
Each of the 768 dimensions in the output vector captures a different learned aspect of the input text's meaning. We don't know exactly what each dimension means — that's the nature of deep learning representations. What we do know is that texts with similar financial meaning will have similar 768-dim vectors, and XGBoost can exploit that structure.

In [12]:
from tqdm.notebook import tqdm

BATCH_SIZE = 64

def get_cls_embeddings(texts, batch_size=BATCH_SIZE):
    """Run FinBERT on a list of texts and return [CLS] embeddings."""
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc='FinBERT inference'):
        batch = texts[i : i + batch_size]

        # Tokenise: convert text to token IDs, pad to same length, truncate if too long
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,       # merchant+category text is short — 64 tokens is plenty
            return_tensors='pt'  # return PyTorch tensors
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}  # move to GPU

        with torch.no_grad():  # no gradient computation needed for inference
            outputs = model(**encoded)

        # Extract [CLS] token: shape is [batch_size, seq_len, 768]
        # Index [:, 0, :] gives us the first token ([CLS]) for each item in the batch
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

    return np.vstack(all_embeddings)  # combine all batches into one array

# Run inference on unique texts only
print(f'Running FinBERT on {len(unique_texts):,} unique text combinations...')
unique_embeddings = get_cls_embeddings(unique_texts)

print(f'\nEmbeddings shape: {unique_embeddings.shape}')
print(f'Each text → {unique_embeddings.shape[1]}-dimensional vector')

Running FinBERT on 700 unique text combinations...


FinBERT inference:   0%|          | 0/11 [00:00<?, ?it/s]


Embeddings shape: (700, 768)
Each text → 768-dimensional vector


## Cell 7 — Build Embedding Lookup and Map to Full Dataset
We have embeddings for the unique texts. Now we map them back to every row in the full 1.3M-row dataset.

**How the merge works:**  
We create a Python dictionary mapping `text_string → 768-dim embedding`. Then for each row in the full dataframe, we look up its text string in the dictionary and retrieve the embedding. This is an O(1) lookup per row — very fast.

In [13]:
# Build lookup: text string → embedding vector
text_to_embedding = dict(zip(unique_texts, unique_embeddings))

print(f'Lookup dictionary built: {len(text_to_embedding):,} entries')

# Map back to every row in the full dataset
print('Mapping embeddings to all rows...')
full_embeddings = np.vstack(
    df['text_input'].map(text_to_embedding).values
)

print(f'Full embeddings shape: {full_embeddings.shape}')
print(f'Covers all {len(df):,} rows: {full_embeddings.shape[0] == len(df)}')

Lookup dictionary built: 700 entries
Mapping embeddings to all rows...
Full embeddings shape: (1296675, 768)
Covers all 1,296,675 rows: True


## Cell 8 — PCA Dimensionality Reduction (768 → 30)
768 dimensions is too many to merge directly with our ~15 tabular features — the embeddings would dominate the feature space and likely hurt model performance.

**Why PCA?**  
PCA (Principal Component Analysis) finds the directions in the 768-dimensional space that capture the most variation in the data. The first 30 principal components typically retain most of the meaningful information while discarding noise. After reduction, the 30 embedding dimensions are on a similar scale to our tabular features and can be merged cleanly.

We fit PCA on the **unique** embeddings only (much faster), then apply the transform to the full dataset.

In [14]:
from sklearn.decomposition import PCA

N_COMPONENTS = 30

# Fit PCA on unique embeddings (fewer rows = faster fit)
pca = PCA(n_components=N_COMPONENTS, random_state=42)
pca.fit(unique_embeddings)

# Report how much variance is retained
explained = pca.explained_variance_ratio_.sum() * 100
print(f'PCA fitted on {len(unique_embeddings):,} unique embeddings.')
print(f'Variance retained by {N_COMPONENTS} components: {explained:.1f}%')
print(f'Per-component variance (first 10):')
for i, v in enumerate(pca.explained_variance_ratio_[:10]):
    print(f'  PC{i+1:02d}: {v*100:.2f}%')

# Apply PCA to unique embeddings (to save as lookup)
pca_unique = pca.transform(unique_embeddings)

# Apply PCA to the full dataset embeddings
pca_full = pca.transform(full_embeddings)

print(f'\nPCA unique shape: {pca_unique.shape}')
print(f'PCA full shape:   {pca_full.shape}')

PCA fitted on 700 unique embeddings.
Variance retained by 30 components: 89.6%
Per-component variance (first 10):
  PC01: 19.70%
  PC02: 12.98%
  PC03: 8.26%
  PC04: 7.64%
  PC05: 6.26%
  PC06: 5.04%
  PC07: 4.27%
  PC08: 3.97%
  PC09: 3.36%
  PC10: 2.75%

PCA unique shape: (700, 30)
PCA full shape:   (1296675, 30)


## Cell 9 — Save Outputs
We save three files that will be downloaded and placed in `data/embeddings/` in VS Code.

**Files to download:**
1. `unique_texts.npy` — the unique text strings (needed for mapping in VS Code)
2. `finbert_embeddings_unique.npy` — raw 768-dim embeddings for unique texts
3. `pca_embeddings_unique.npy` — 30-dim PCA embeddings for unique texts

We save the unique-text versions (not the full 1.3M-row arrays) because they are much smaller and Cell 10 will reconstruct the full mapping locally.

In [15]:
# Save as .npy files (NumPy binary format — compact and fast to load)
np.save('unique_texts.npy',             np.array(unique_texts))
np.save('finbert_embeddings_unique.npy', unique_embeddings)
np.save('pca_embeddings_unique.npy',    pca_unique)

print('Files saved:')
print(f'  unique_texts.npy             — {len(unique_texts):,} strings')
print(f'  finbert_embeddings_unique.npy — shape {unique_embeddings.shape}')
print(f'  pca_embeddings_unique.npy     — shape {pca_unique.shape}')

# Download all three files to your computer
print('\nDownloading files...')
from google.colab import files
files.download('unique_texts.npy')
files.download('finbert_embeddings_unique.npy')
files.download('pca_embeddings_unique.npy')
print('Done. All three files in this folder: data/embeddings/')

Files saved:
  unique_texts.npy             — 700 strings
  finbert_embeddings_unique.npy — shape (700, 768)
  pca_embeddings_unique.npy     — shape (700, 30)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done. All three files in this folder: data/embeddings/


---
# ⬇️ CELL OUTPUTS RAN IN COLAB
Download the three `.npy` files and place them in:
```
fraud-detection-project/
└── data/
    └── embeddings/
        ├── unique_texts.npy
        ├── finbert_embeddings_unique.npy
        └── pca_embeddings_unique.npy
```
Then **Cell 10 only** will be run in VS CODE.

---

## Cell 10 — Merge Embeddings into Feature Sets (Run in VS Code)
Now we are back in VS Code. We load the three `.npy` files from `data/embeddings/`, reconstruct the lookup table, and merge the 30 PCA dimensions into `X_train` and `X_test`.

**How the merge works:**  
1. Load `fraudTrain.csv` to get the original merchant + category columns
2. Re-create the same text strings as Colab did
3. Look up each row's PCA embedding from the dictionary
4. Apply the same time-based sort and 80/20 split as notebook 02
5. Add `embed_0` through `embed_29` columns to `X_train` and `X_test`
6. Overwrite the saved CSV files with the enriched versions

In [1]:
# RUN IN VS CODE — after placing .npy files in data/embeddings/
import pandas as pd
import numpy as np
from pathlib import Path

DATA_RAW        = Path('../data/raw')
DATA_PROCESSED  = Path('../data/processed')
DATA_EMBEDDINGS = Path('../data/embeddings')

# Load the .npy files saved from Colab
unique_texts      = np.load(DATA_EMBEDDINGS / 'unique_texts.npy', allow_pickle=True).tolist()
pca_unique        = np.load(DATA_EMBEDDINGS / 'pca_embeddings_unique.npy')

print(f'Unique texts loaded:  {len(unique_texts):,}')
print(f'PCA embeddings shape: {pca_unique.shape}')

# Build lookup: text → 30-dim PCA embedding
text_to_pca = dict(zip(unique_texts, pca_unique))

# Reload raw data and re-create text strings (same logic as Colab Cell 3)
df = pd.read_csv(DATA_RAW / 'fraudTrain.csv')
df['trans_datetime'] = pd.to_datetime(df['trans_date_trans_time'])
df['text_input'] = (
    'merchant: ' + df['merchant'].str.replace('fraud_', '', regex=False) +
    ', category: ' + df['category']
)

# Map PCA embeddings to every row
pca_full = np.vstack(df['text_input'].map(text_to_pca).values)
print(f'Full PCA array shape: {pca_full.shape}')

# Add embed_0 ... embed_29 columns
embed_cols = [f'embed_{i}' for i in range(pca_full.shape[1])]
embed_df   = pd.DataFrame(pca_full, columns=embed_cols, index=df.index)
df = pd.concat([df, embed_df], axis=1)

# Replicate notebook 02 sort + split (same order as X_train/X_test)
df.sort_values(['cc_num', 'trans_datetime'], inplace=True)
df.reset_index(drop=True, inplace=True)

split_idx = int(len(df) * 0.80)
train_df  = df.iloc[:split_idx]
test_df   = df.iloc[split_idx:]

# Load existing X_train / X_test and append embedding columns
X_train = pd.read_csv(DATA_PROCESSED / 'X_train.csv')
X_test  = pd.read_csv(DATA_PROCESSED / 'X_test.csv')

# Add embedding columns
X_train = pd.concat([X_train.reset_index(drop=True),
                     train_df[embed_cols].reset_index(drop=True)], axis=1)
X_test  = pd.concat([X_test.reset_index(drop=True),
                     test_df[embed_cols].reset_index(drop=True)],  axis=1)

# Save enriched feature sets
X_train.to_csv(DATA_PROCESSED / 'X_train.csv', index=False)
X_test.to_csv(DATA_PROCESSED  / 'X_test.csv',  index=False)

print(f'\nX_train shape (with embeddings): {X_train.shape}')
print(f'X_test shape  (with embeddings): {X_test.shape}')
print(f'\nEmbedding columns added: {embed_cols[:5]} ... {embed_cols[-1]}')
print('\nSaved enriched X_train.csv and X_test.csv to data/processed/')
print('\nVerification — first 3 embed columns of X_train:')
print(X_train[embed_cols[:3]].head(3))

Unique texts loaded:  700
PCA embeddings shape: (700, 30)
Full PCA array shape: (1296675, 30)

X_train shape (with embeddings): (1037340, 46)
X_test shape  (with embeddings): (259335, 46)

Embedding columns added: ['embed_0', 'embed_1', 'embed_2', 'embed_3', 'embed_4'] ... embed_29

Saved enriched X_train.csv and X_test.csv to data/processed/

Verification — first 3 embed columns of X_train:
    embed_0   embed_1   embed_2
0 -1.010430 -1.241044  0.822944
1  0.482262 -1.732923 -1.660964
2  0.526554 -1.625211 -1.582646
